# Qualidade da camada Silver

Verificações usadas antes da criação da Gold.

In [0]:
from pyspark.sql import functions as F

partidas = spark.table("workspace.silver.partidas")
gols = spark.table("workspace.silver.gols")
cartoes = spark.table("workspace.silver.cartoes")
estatisticas = spark.table("workspace.silver.estatisticas")

## Contagens

In [0]:
contagens = [
    ("partidas", partidas.count()),
    ("gols", gols.count()),
    ("cartoes", cartoes.count()),
    ("estatisticas", estatisticas.count()),
]

display(spark.createDataFrame(contagens, ["tabela", "registros"]))

## Chaves e relacionamentos

In [0]:
ids_partidas = partidas.select("partida_id")

duplicadas = partidas.groupBy("partida_id").count().filter("count > 1").count()
gols_orfaos = gols.join(ids_partidas, "partida_id", "left_anti").count()
cartoes_orfaos = cartoes.join(ids_partidas, "partida_id", "left_anti").count()
estatisticas_orfas = estatisticas.join(ids_partidas, "partida_id", "left_anti").count()
estatisticas_fora_do_grao = (
    estatisticas.groupBy("partida_id").count().filter("count <> 2").count()
)

integridade = [
    ("IDs duplicados em partidas", duplicadas),
    ("Gols sem partida", gols_orfaos),
    ("Cartões sem partida", cartoes_orfaos),
    ("Estatísticas sem partida", estatisticas_orfas),
    ("Partidas sem duas linhas de estatísticas", estatisticas_fora_do_grao),
]

display(spark.createDataFrame(integridade, ["teste", "ocorrencias"]))

## Campos principais

In [0]:
campos = [
    ("partida_id nulo", partidas.filter("partida_id IS NULL").count()),
    ("data nula", partidas.filter("data_partida IS NULL").count()),
    ("mandante vazio", partidas.filter("mandante IS NULL OR trim(mandante) = ''").count()),
    ("visitante vazio", partidas.filter("visitante IS NULL OR trim(visitante) = ''").count()),
    ("placar negativo", partidas.filter("placar_mandante < 0 OR placar_visitante < 0").count()),
    ("posse inválida", estatisticas.filter("posse_bola_pct < 0 OR posse_bola_pct > 100").count()),
    ("precisão inválida", estatisticas.filter("precisao_passes_pct < 0 OR precisao_passes_pct > 100").count()),
    ("minuto de gol inválido", gols.filter("minuto_base IS NULL").count()),
    ("minuto de cartão inválido", cartoes.filter("minuto_base IS NULL").count()),
]

display(spark.createDataFrame(campos, ["teste", "ocorrencias"]))

## Cobertura das estatísticas

In [0]:
cobertura = (
    estatisticas.join(partidas.select("partida_id", F.year("data_partida").alias("ano")), "partida_id")
    .groupBy("ano")
    .agg(
        F.count("*").alias("linhas"),
        F.count("posse_bola_pct").alias("com_posse"),
        F.count("precisao_passes_pct").alias("com_precisao_passes"),
        F.sum(F.col("estatisticas_disponiveis").cast("int")).alias("disponiveis")
    )
    .withColumn("posse_ausente", F.col("linhas") - F.col("com_posse"))
    .orderBy("ano")
)

display(cobertura)

## Placar e eventos de gols

In [0]:
primeira_partida_com_eventos = gols.agg(F.min("partida_id")).first()[0]

gols_por_clube = gols.groupBy("partida_id", "clube").count()

placares = (
    partidas.filter(F.col("partida_id") >= primeira_partida_com_eventos).alias("p")
    .join(
        gols_por_clube.alias("gm"),
        (F.col("p.partida_id") == F.col("gm.partida_id")) & (F.col("p.mandante") == F.col("gm.clube")),
        "left"
    )
    .join(
        gols_por_clube.alias("gv"),
        (F.col("p.partida_id") == F.col("gv.partida_id")) & (F.col("p.visitante") == F.col("gv.clube")),
        "left"
    )
    .select(
        F.col("p.partida_id"),
        F.col("p.placar_mandante"),
        F.col("p.placar_visitante"),
        F.coalesce(F.col("gm.count"), F.lit(0)).alias("gols_mandante"),
        F.coalesce(F.col("gv.count"), F.lit(0)).alias("gols_visitante")
    )
)

placares_divergentes = placares.filter(
    (F.col("placar_mandante") != F.col("gols_mandante")) |
    (F.col("placar_visitante") != F.col("gols_visitante"))
)

resumo_placar = [
    ("Primeiro ID com eventos", primeira_partida_com_eventos),
    ("Partidas verificadas", placares.count()),
    ("Placares divergentes", placares_divergentes.count()),
]

display(spark.createDataFrame(resumo_placar, ["medida", "valor"]))

## Resultado

In [0]:
falhas_criticas = sum(valor for _, valor in integridade + campos) + placares_divergentes.count()

resultado = "aprovada" if falhas_criticas == 0 else "revisar"
display(spark.createDataFrame([(resultado, falhas_criticas)], ["situacao", "falhas_criticas"]))